In [122]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

In [2]:
qqp = pd.read_csv('raw_data/07-2026_Q2.csv')

In [3]:
#Escoger Food_categories
FOOD_CATEGORIES = [
    "Hortalizas Frescas", "Frutas Frescas", "Frutas y Legumbres Procesadas",
    "Legumbres Secas", "Condimentos", "Carnes Frías Secas y Embutidos",
    "Carne y Vísceras de Res", "Carne y Vísceras de Cerdo", "Carne de Ave",
    "Pescados y Mariscos", "Pescados y Mariscos en Conserva", "Huevo",
    "Derivados de Leche", "Leche Procesada", "Leche Fresca",
    "Galletas Pastas y Harinas de Trigo", "Arroz y Cereales Preparados",
    "Tortillas y Derivados del Maíz", "Pan",
    "Aceites y Grasas Veg. Comestibles", "Grasas Animales Comestibles",
    "Azúcar", "Café", "Té", "Vinos y Licores",
]


In [4]:
#Incluir Food_categories
EXCLUDE_PRODUCTS = [
    "Pastelillos y Pan Dulce Empaquetado", "Galletas Dulces", "Galletas Saladas",
    "Galletas Populares", "Barras de Cereal", "Barras de Avena", "Granola",
    "Cereales", "Helado", "Mermelada", "Maíz Palomero", "Sustituto de Azúcar",
    "Harina Hot Cakes", "Empanizadores",
    "Alimentos Preparados", "Alimento Preparado para Niños", "Fórmula Láctea",
    "Producto Lácteo", "Barrita de Surimi", "Queso de Puerco",
    "Mezcla California", "Combinado Primavera", "Mezcla Campesina",
    "Guarnición de Verduras", "Pastel Pimiento", "Floretes de Brócoli",
    "Tequila", "Brandy", "Whisky", "Ron",
]


In [5]:
COLLAPSE = {
    "Frijoles": "Frijol",
    "Granos de Elote": "Elote",
    "Chícharos en Lata": "Chícharos",
    "Chícharo": "Chícharos",
    "Sal de Mar": "Sal Molida de Mesa",
    "Café Tostado y Molido": "Café Soluble",
    "Leche Pasteurizada": "Leche Ultrapasteurizada",
    "Tilapia": "Mojarra",
    "Basa": "Mojarra",
}

In [19]:
food_categories = qqp[qqp["categoria"].isin(FOOD_CATEGORIES)]

print(len(food), "of", len(qqp), "rows")

287484 of 744623 rows


In [21]:
food_categories.shape

(287484, 15)

In [26]:
food_presentation_list = food_categories['presentacion'].value_counts(normalize=True)

presentacion
1 Kg. Granel                                     0.044086
Paquete 400 Gr.                                  0.014578
Barra 90 Gr. Sin Sal                             0.011190
Lata 425 Gr. en Tomate                           0.010449
Caja 1 Lt. Entera                                0.010056
                                                   ...   
1 Kg. Granel. Morena Americana                   0.000003
Paquete Mediano 454 Gr. Rebanado Blanco          0.000003
1 Kg. Granel. de Pierna (Mandolina). Horneado    0.000003
1 Kg. Granel. Entero. Salmonada. Fresco          0.000003
Charola 250 Gr. Seco                             0.000003
Name: proportion, Length: 927, dtype: float64

In [31]:
food_categories[["quantity", "unit"]] = (
    food_categories["presentacion"]
    .str.extract(r"(\d+(?:[.,]\d+)?)\s*(kg|gr|g|lt|l)\b", flags=re.IGNORECASE)
)

food_categories["quantity"] = (
    food_categories["quantity"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

food_categories["unit"] = food_categories["unit"].str.lower()

/var/folders/jc/syl_hqg13ld03t1lw1h83t200000gn/T/ipykernel_3632/3016665531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  food_categories[["quantity", "unit"]] = (
/var/folders/jc/syl_hqg13ld03t1lw1h83t200000gn/T/ipykernel_3632/3016665531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  food_categories[["quantity", "unit"]] = (
/var/folders/jc/syl_hqg13ld03t1lw1h83t200000gn/T/ipykernel_3632/3016665531.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
T

In [46]:
# Convert kg quantities to grams
food_categories.loc[
    food_categories["unit"].eq("kg"),
    "quantity"
] *= 1000

# Rename kg to gr
food_categories["unit"] = food_categories["unit"].replace({
    "kg": "gr",
    "g": "gr"
})

In [47]:
food_categories

,producto,presentacion,marca,categoria,catalogo,precio,fecha_registro,cadena_comercial,giro,nombre_comercial,direccion,estado,municipio,latitud,longitud,quantity,unit
0,Acelga,Manojo,S/M,Hortalizas Frescas,Frutas y Legumbres,20.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,Av. Mahatma Gandhi S/N. Salida a México. Col. ...,Aguascalientes,Aguascalientes,21.832072,-102.292976,NaN,NaN
1,Aguacate,1 Kg. Granel. Hass,S/M,Frutas Frescas,Frutas y Legumbres,49.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,Av. Mahatma Gandhi S/N. Salida a México. Col. ...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr
2,Ajo,1 Kg. Granel. o Ajo Morado,S/M,Hortalizas Frescas,Frutas y Legumbres,140.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,Av. Mahatma Gandhi S/N. Salida a México. Col. ...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr
3,Apio,1 Kg. Granel,S/M,Hortalizas Frescas,Frutas y Legumbres,35.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,Av. Mahatma Gandhi S/N. Salida a México. Col. ...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr
4,Betabel,1 Kg. Granel,S/M,Hortalizas Frescas,Frutas y Legumbres,35.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,Av. Mahatma Gandhi S/N. Salida a México. Col. ...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
744612,Pan Dulce,Bisquet. Pieza de 68 a 90 Gr. (11 Cm. de Diáme...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,Av. Heroico Colegio Militar Ote. 4-a. Entre S....,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr
744613,Pan Dulce,Concha. Pieza de 68 a 90 Gr. (12 Cm. de Diámet...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,Av. Heroico Colegio Militar Ote. 4-a. Entre S....,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr
744614,Pan Dulce,Cuerno. Pieza de 68 a 90 Gr. (13 Cm. de Diámet...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,Av. Heroico Colegio Militar Ote. 4-a. Entre S....,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr
744615,Pan Dulce,Dona Cubierta de Chocolate. Pieza de 68 a 82 G...,S/M,Pan,Básicos,15.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,Av. Heroico Colegio Militar Ote. 4-a. Entre S....,Zacatecas,Zacatecas,22.750343,-102.519113,82.0,gr


In [55]:
food_categories.isnull().sum(), food_categories['unit'].value_counts()

(producto                0
 presentacion            0
 marca                   0
 categoria               0
 catalogo                0
 precio                  0
 fecha_registro          0
 cadena_comercial        0
 giro                    0
 nombre_comercial        0
 direccion               0
 estado                  0
 municipio               0
 latitud               151
 longitud              151
 quantity            26681
 unit                26681
 dtype: int64,
 unit
 gr    249421
 lt     11382
 Name: count, dtype: int64)

In [44]:
food_categories[
    food_categories["quantity"].isna()
]['producto'].value_counts()

producto
Huevo                        5744
Aceite                       5188
Tequila                      1644
Salsa Inglesa                1602
Salsa Picante y Similares    1173
Lechuga                      1040
Crema                        1032
Brandy                        919
Whisky                        915
Ron                           839
Leche Ultrapasteurizada       831
Pan Blanco Bolillo            691
Cilantro                      629
Espinacas                     592
Elote                         547
Coliflor                      535
Cebolla                       529
Rábano                        512
Acelga                        486
Salsa de Soya                 374
Crema Batida                  287
Vinagre                       263
Vainilla                      158
Verdolaga                      91
Camarón                        29
Alcachofa                      26
Galletas Dulces                 5
Name: count, dtype: int64

In [50]:
food_categories[
    food_categories["producto"].eq("Huevo")
]['presentacion'].value_counts()

presentacion
Paquete C/18 Blanco             2870
Paquete C/12 Blanco             1642
Paquete C/30 Blanco              799
Paquete C/18 Rojo                210
Paquete C/12 Rojo                204
1 Kg. Granel. Blanco              98
Paquete C/18 Blanco. Jumbo        14
Paquete C/18 Blanco. Mediano       3
1 Kg. Granel. Rojo                 2
Cartera C/30 Blanco                2
Name: count, dtype: int64

In [51]:
#Remove eggs with KG, leave the ones using egg counts
rows_to_remove = (
    food_categories["producto"].eq("Huevo")
    & food_categories["presentacion"].str.contains(
        r"\bkg\b",
        case=False,
        na=False
    )
)

food_categories_remove_egg_kg = food_categories[~rows_to_remove].copy()

In [56]:
food_categories_remove_egg_kg[
    food_categories_remove_egg_kg["producto"].eq("Huevo")
]['presentacion'].value_counts()

presentacion
Paquete C/18 Blanco             2870
Paquete C/12 Blanco             1642
Paquete C/30 Blanco              799
Paquete C/18 Rojo                210
Paquete C/12 Rojo                204
Paquete C/18 Blanco. Jumbo        14
Paquete C/18 Blanco. Mediano       3
Cartera C/30 Blanco                2
Name: count, dtype: int64

In [57]:
egg_mask = food_categories_remove_egg_kg["producto"].eq("Huevo")

food_categories_remove_egg_kg.loc[egg_mask, "quantity"] = (
    food_categories_remove_egg_kg.loc[egg_mask, "presentacion"]
    .str.extract(r"C/\s*(\d+)", expand=False)
    .astype(float)
)

food_categories_remove_egg_kg.loc[egg_mask, "unit"] = "piece"

In [60]:
food_categories_remove_egg_kg[food_categories_remove_egg_kg['producto']=='Huevo']['quantity'].value_counts()

quantity
18.0    3097
12.0    1846
30.0     801
Name: count, dtype: int64

In [63]:
food_categories_remove_egg_kg[
    food_categories_remove_egg_kg["quantity"].isna()
]['producto'].value_counts()

producto
Aceite                       5188
Tequila                      1644
Salsa Inglesa                1602
Salsa Picante y Similares    1173
Lechuga                      1040
Crema                        1032
Brandy                        919
Whisky                        915
Ron                           839
Leche Ultrapasteurizada       831
Pan Blanco Bolillo            691
Cilantro                      629
Espinacas                     592
Elote                         547
Coliflor                      535
Cebolla                       529
Rábano                        512
Acelga                        486
Salsa de Soya                 374
Crema Batida                  287
Vinagre                       263
Vainilla                      158
Verdolaga                      91
Camarón                        29
Alcachofa                      26
Galletas Dulces                 5
Name: count, dtype: int64

In [67]:
food_categories[
    food_categories["producto"].eq("Elote")
]['presentacion'].value_counts()

presentacion
Lata 220 Gr. en Grano    1757
Lata 400 Gr. en Grano    1176
Lata 225 Gr. en Grano     560
Pieza                     547
Name: count, dtype: int64

In [69]:
food_categories_clean_elote = food_categories_remove_egg_kg.copy()

elote_mask = food_categories_clean_elote["producto"].eq("Elote")

# Extract the quantity for Elote products measured in grams
food_categories_clean_elote.loc[elote_mask, "quantity"] = (
    food_categories_clean_elote.loc[elote_mask, "presentacion"]
    .str.extract(r"(\d+(?:[.,]\d+)?)\s*Gr\.?", expand=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# Set the unit only for successfully extracted Elote rows
food_categories_clean_elote.loc[
    elote_mask & food_categories_clean_elote["quantity"].notna(),
    "unit"
] = "gr"

food_categories_clean_elote = food_categories_clean_elote[
    ~(
        food_categories_clean_elote["producto"].eq("Elote")
        & food_categories_clean_elote["quantity"].isna()
    )
].copy()

In [75]:
food_categories_clean_elote[
    food_categories_clean_elote["quantity"].isna()
]['producto'].value_counts()

producto
Aceite                       5188
Tequila                      1644
Salsa Inglesa                1602
Salsa Picante y Similares    1173
Lechuga                      1040
Crema                        1032
Brandy                        919
Whisky                        915
Ron                           839
Leche Ultrapasteurizada       831
Pan Blanco Bolillo            691
Cilantro                      629
Espinacas                     592
Coliflor                      535
Cebolla                       529
Rábano                        512
Acelga                        486
Salsa de Soya                 374
Crema Batida                  287
Vinagre                       263
Vainilla                      158
Verdolaga                      91
Camarón                        29
Alcachofa                      26
Galletas Dulces                 5
Name: count, dtype: int64

In [88]:
#Remove NA after inspecting the NA values, most of the products will not be used in the prediction model
food_categories_clean = food_categories_clean_elote.dropna(
    subset=["quantity", "unit"]
).copy()


In [89]:
food_categories_clean.shape

(266447, 17)

In [111]:
#Add an price column to calculate the price
food_categories_clean["price_per_unit"] = (
    food_categories_clean["precio"]
    / food_categories_clean["quantity"]
)

food_categories_clean

,producto,presentacion,marca,categoria,catalogo,precio,fecha_registro,cadena_comercial,giro,nombre_comercial,...,estado,municipio,latitud,longitud,quantity,unit,min_price,max_price,average_price,price_per_unit
0,Aguacate,1 Kg. Granel. Hass,S/M,Frutas Frescas,Frutas y Legumbres,49.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr,32.00,110.0,65.59,0.049000
1,Ajo,1 Kg. Granel. o Ajo Morado,S/M,Hortalizas Frescas,Frutas y Legumbres,140.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr,40.90,299.0,95.53,0.140000
2,Apio,1 Kg. Granel,S/M,Hortalizas Frescas,Frutas y Legumbres,35.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr,19.90,45.0,34.43,0.035000
3,Betabel,1 Kg. Granel,S/M,Hortalizas Frescas,Frutas y Legumbres,35.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr,17.82,50.0,34.57,0.035000
4,Brócoli,1 Kg. Granel,S/M,Hortalizas Frescas,Frutas y Legumbres,22.0,2026/07/16,Central de Abasto,Central de Abasto,Central de Abasto,...,Aguascalientes,Aguascalientes,21.832072,-102.292976,1000.0,gr,17.00,70.0,51.10,0.022000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266442,Pan Dulce,Bisquet. Pieza de 68 a 90 Gr. (11 Cm. de Diáme...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,...,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr,8.00,36.0,11.84,0.133333
266443,Pan Dulce,Concha. Pieza de 68 a 90 Gr. (12 Cm. de Diámet...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,...,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr,8.00,36.0,11.84,0.133333
266444,Pan Dulce,Cuerno. Pieza de 68 a 90 Gr. (13 Cm. de Diámet...,S/M,Pan,Básicos,12.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,...,Zacatecas,Zacatecas,22.750343,-102.519113,90.0,gr,8.00,36.0,11.84,0.133333
266445,Pan Dulce,Dona Cubierta de Chocolate. Pieza de 68 a 82 G...,S/M,Pan,Básicos,15.0,2026/07/31,Panaderias Tradicionales,Panaderías,Panificadora Remedio Sucursal Colegio Militar,...,Zacatecas,Zacatecas,22.750343,-102.519113,82.0,gr,8.00,36.0,11.84,0.182927


In [119]:
food_categories_clean[food_categories_clean['producto']=='Huevo']['price_per_unit'].value_counts()

price_per_unit
2.333333    707
2.388889    468
2.250000    362
2.055556    347
2.111111    296
           ... 
3.161111      1
2.253333      1
2.958333      1
2.422222      1
2.016667      1
Name: count, Length: 177, dtype: int64

In [115]:
#Calculate Price Summary for each unique product
price_summary = (
    food_categories_clean
    .groupby(["producto","unit"], as_index=False)
    .agg(
        min_price=("price_per_unit", "min"),
        max_price=("price_per_unit", "max"),
        average_price=("price_per_unit", "mean"),
    ).round(2)
)


price_summary

,producto,unit,min_price,max_price,average_price
0,Aceite,lt,34.50,72.50,44.69
1,Aguacate,gr,0.03,0.11,0.07
2,Ajo,gr,0.07,0.78,0.39
3,Ajonjolí,gr,0.04,0.24,0.16
4,Alcachofa,gr,0.10,0.18,0.15
...,...,...,...,...,...
199,Verdolaga,gr,0.02,0.10,0.05
200,Villajaiba,gr,0.12,0.20,0.16
201,Vinagre,lt,9.00,29.90,16.66
202,Yoghurt,gr,0.03,0.40,0.08


In [124]:
#Remove accent mark from Spanish
price_summary["producto"] = (
    price_summary["producto"]
    .apply(
        lambda text: "".join(
            character
            for character in unicodedata.normalize("NFD", text)
            if unicodedata.category(character) != "Mn"
        )
    )
)

In [125]:
price_summary

,producto,unit,min_price,max_price,average_price
0,Aceite,lt,34.50,72.50,44.69
1,Aguacate,gr,0.03,0.11,0.07
2,Ajo,gr,0.07,0.78,0.39
3,Ajonjoli,gr,0.04,0.24,0.16
4,Alcachofa,gr,0.10,0.18,0.15
...,...,...,...,...,...
199,Verdolaga,gr,0.02,0.10,0.05
200,Villajaiba,gr,0.12,0.20,0.16
201,Vinagre,lt,9.00,29.90,16.66
202,Yoghurt,gr,0.03,0.40,0.08


In [126]:
price_summary.to_csv('precio_por_producto.csv')